# Script for comparing different CNN models 

In [1]:
import os.path as op
import mne 
import os
from termcolor import colored
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,mean_absolute_error,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle
import tensorflow as tf

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 


from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, Conv1DTranspose, MaxPooling1D, Flatten, Dense, Input, Dropout,BatchNormalization  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
mne.set_log_level("CRITICAL")

# Loading in Data

In [2]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration_1205.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration_1', 'Duration_2']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

features_all = features_all.rename(columns={
    'Duration_1': 'Duration_corr',
    'Duration_2': 'Duration_zygo'
})

features_all_store = features_all

x = np.isnan(features_all['Duration_zygo']) 
indices = np.where(x)[0]
features_all = features_all.drop(indices)

In [ ]:
np.shape(features_all_store)
 

# Functions

In [9]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?
    model.add(MaxPooling1D(pool_size=1)) 

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [ ]:
# single head CNN model for duration 
def CNN_model_duration(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(1, activation='linear'))


    return model  # Return the compiled model

In [48]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)


    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


    shared = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    shared = Dropout(0.5)(shared)

    # Count-specific hidden layers
    count_branch = Dense(32, activation='relu')(shared)
    count_branch = Dropout(0.2)(count_branch)

    count_output = Dense(
        num_classes,
        activation='softmax',
        name='count_output'
    )(count_branch)

    duration_branch = Dense(32, activation='relu')(shared)
    duration_branch = Dropout(0.2)(duration_branch) # try increasing drop out for more regularization? (increase if overfitting)

    duration_output = Dense(
        1,
        activation='linear',
        name='duration_output'
    )(duration_branch)


    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [ ]:
# single head CNN model for number of contractions  
def CNN_model_contraction_multichan(input_shape, num_classes,feature_num):
    global epoch_len
    
    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(16, kernel_size=3, activation='relu', input_shape=input_shape, kernel_regularizer=l2(0.0001))(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer
    x = Conv1D(32, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)



    x = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.7)(x)

    x = Flatten()(x)

    # two heads for each label 
    # output head for Zygo
    out_zygo = Dense(num_classes, activation='softmax', name="zygo_output", kernel_regularizer=l2(0.001))(x)

    # output head for Corr
    out_corr = Dense(num_classes, activation='softmax', name="corr_output", kernel_regularizer=l2(0.001))(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])
    
    return model 

In [ ]:
def CNN_model_duration_multichan(input_shape, num_classes, feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(32, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(64, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)

    x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.4)(x)

    # output heads
    out_zygo = Dense(1, activation='relu', name="zygo_output")(x)
    out_corr = Dense(1, activation='relu', name="corr_output")(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])

    return model

In [ ]:
def CNN_model_fourhead_multichan(input_shape, num_classes, feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # --- Shared CNN backbone ---
    x = Conv1D(32, kernel_size=3, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)
    shared = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    shared = Dropout(0.5)(shared)

    # zygo branch 
    zygo_branch = Dense(32, activation='relu')(shared)
    zygo_branch = Dropout(0.3)(zygo_branch)
    zygo_count_output    = Dense(num_classes, activation='softmax', name='zygo_count_output')(zygo_branch)
    zygo_duration_output = Dense(1, activation='linear', name='zygo_duration_output')(zygo_branch)

    # corr branch 
    corr_branch = Dense(32, activation='relu')(shared)
    corr_branch = Dropout(0.3)(corr_branch)
    corr_count_output    = Dense(num_classes, activation='softmax', name='corr_count_output')(corr_branch)
    corr_duration_output = Dense(1, activation='linear', name='corr_duration_output')(corr_branch)

    # --- Model ---
    model = Model(
        inputs=inputs,
        outputs=[
            zygo_count_output,
            zygo_duration_output,
            corr_count_output,
            corr_duration_output
        ]
    )

    return model

# Single Channel Training 

## Single Head Count 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_zygo = features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()

indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_contraction=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_contraction = CNN_model_contraction(input_shape, num_classes,feature_num)
    model_contraction.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

    early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_contraction.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_contraction.evaluate(X_val,y_val)
    cvScores_contraction.append(scores[1] * 100)

    k += 1 

# redefine fresh model 
#final_model = CNN_model_contraction(input_shape, num_classes, feature_num)
#final_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])

model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test), callbacks=[early_stop])

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores_contraction)
stdScores = np.std(cvScores_contraction)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

# full training results (test data not seen during cross val)
y_pred_train = model_contraction.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model_contraction.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   


# Calculate accuracy
accuracy_training = accuracy_score(y_train_full, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train_full, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Model scores---------------")
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test) 

# Count by Muscle Groups 

In [ ]:
zygo_ind_train = np.where(idx_train < 7140)
corr_ind_train = np.where(idx_train >= 7140)  
zygo_ind_test = np.where(idx_test < 7140)
corr_ind_test = np.where(idx_test >= 7140)  

In [ ]:
# remake splits (i think this is wrong)
X_train_full_corr = X[corr_ind_train]
X_train_full_zygo = X[zygo_ind_train]

X_test_corr = X[corr_ind_test]
X_test_zygo = X[zygo_ind_test

y_train_full_corr = y[corr_ind_train]
y_train_full_zygo = y[zygo_ind_train]

y_test_corr = y[corr_ind_test]
y_test_zygo = y[zygo_ind_test]


# full training results (test data not seen during cross val)
y_pred_train_corr = model_contraction.predict(X_train_full_corr)  
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)   

y_pred_train_zygo = model_contraction.predict(X_train_full_zygo)  
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)   

# Predict on test data
y_pred_test_zygo = model_contraction.predict(X_test_zygo)   
y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   

y_pred_test_corr= model_contraction.predict(X_test_corr)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   

# Calculate accuracy
accuracy_training_corr = accuracy_score(y_train_full_corr, y_pred_train_corr)   
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)  

accuracy_training_zygo = accuracy_score(y_train_full_zygo, y_pred_train_zygo)   
accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)  

# Calculate F1 score
f1_training_corr = f1_score(y_train_full_corr, y_pred_train_corr, average='weighted')  
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')  

f1_training_zygo = f1_score(y_train_full_zygo, y_pred_train_zygo, average='weighted')  
f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')  


# Print accuracy and F1 score
print("Corru Scores------------------------------")  
print("Training Accuracy :", accuracy_training_corr) 
print("Test Accuracy :", accuracy_test_corr)  
print("Training F1 Score :", f1_training_corr)  
print("Test F1 Score :", f1_test_corr) 

print("Zygo Scores------------------------------")  
print("Training Accuracy :", accuracy_training_zygo)  
print("Test Accuracy :", accuracy_test_zygo)  
print("Training F1 Score :", f1_training_zygo)   
print("Test F1 Score :", f1_test_zygo) 

In [ ]:
corr_mask = muscle_test == "Corr"
zygo_mask = muscle_test == "Zygo"

print("\nCorr Results-------------------")
#print("Accuracy Train:", accuracy_score(y_train_full[corr_mask], y_pred_train[corr_mask]))
print("Accuracy Test:", accuracy_score(y_test[corr_mask], y_pred_test[corr_mask]))

#print("F1 Train:", f1_score(y_train_full[corr_mask], y_pred_train[corr_mask]))
print("F1 Test:", f1_score(y_test[corr_mask], y_pred_test[corr_mask], average='weighted')  )

print("\nZygo Results-------------------")
#print("Accuracy Train:", accuracy_score(y_train_full[zygo_mask], y_pred_train[zygo_mask]))
print("Accuracy Test:", accuracy_score(y_test[zygo_mask], y_pred_test[zygo_mask])  )

#print("F1 Train:", f1_score(y_train_full[zygo_mask], y_pred_train[zygo_mask]))
print("F1 Test:", f1_score(y_test[zygo_mask], y_pred_test[zygo_mask], average='weighted')  )


# Single Head Duration

In [11]:
# Define CNN model inputs
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all

muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = (np.concatenate((X_zygo, X_corr), axis=0))
y = (np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]]))     


indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
 KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_duration=[]
epoch_num = 100 
k = 1
 
for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]


    model_duration = CNN_model_duration(input_shape, num_classes,feature_num)
    model_duration.compile(optimizer='adam', loss='mae', metrics=['mae'])  

    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model_duration.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_duration.evaluate(X_val,y_val)
    cvScores_duration.append(scores[1])

    k += 1 kf =
    

model_history_duration = model_duration.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test), callbacks=[early_stop])

In [ ]:
# cross validation results 
cvScores_duration = np.array(cvScores_duration)

avgScores = np.mean(cvScores_duration)
stdScores = np.std(cvScores_duration)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
 
baseline = np.mean(y_train_full)
mae_baseline = np.mean(np.abs(y_train_full - baseline))

# full training results (test data not seen during cross val)
y_pred_train_dur = model_duration.predict(X_train_full) 

# Predict on test data
y_pred_test_dur = model_duration.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full, y_pred_train_dur)   
mae_test_dur= mean_absolute_error(y_test, y_pred_test_dur)  
  
# Print accuracy and F1 score
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)

corr_mask = muscle_test == "Corr"
zygo_mask = muscle_test == "Zygo"

print("\nCorr Results-------------------")
print("MAE:", mean_absolute_error(y_test[corr_mask], y_pred_test_dur[corr_mask]))

print("\nZygo Results-------------------")
print("MAE:", mean_absolute_error(y_test[zygo_mask], y_pred_test_dur[zygo_mask]))
 

# Two Head 

In [15]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all

muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [49]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 

cvScores_dur =[]
cvScores_contr =[]
cvScores = [] 

epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_twohead = CNN_model_twohead(input_shape, num_classes,feature_num) #CNN_model_regression if treating both as continuous
    model_twohead.compile(
                optimizer="adam",
                loss={
                    "count_output": "sparse_categorical_crossentropy",  # or categorical_crossentropy
                    "duration_output": "mae",
                },
                metrics={
                    "count_output": ["accuracy"],
                    "duration_output": ["mae"],
                }
            ) #think about metric 

    model_history_kfold = model_twohead.fit(
    X_train,
    {
        "count_output": y_train[:, 0],
        "duration_output": y_train[:, 1],
    },
    validation_data=(
        X_val,
        {
            "count_output": y_val[:, 0],
            "duration_output": y_val[:, 1],
        }
    ),
    epochs=epoch_num
    )

    scores = model_twohead.evaluate(
        X_val,
        {
            "count_output": y_val[:, 0],
            "duration_output": y_val[:, 1],
        },
        return_dict=True
    )



    cvScores_dur.append(metrics['duration_output_mae'])
    cvScores_contr.append(metrics['count_output_accuracy'] * 100)


    cvScores.append(scores)
 
    
    

    k += 1 
    

model_history_twohead = model_twohead.fit(X_train_full, [y_train_full[:,0], y_train_full[:,1]], epochs=epoch_num, validation_data=(X_test, [y_test[:,0], y_test[:,1]])) #, callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/10


2026-05-12 23:10:31.846079: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 39s 98ms/step - loss: 45.7850 - count_output_loss: 3.4972 - duration_output_loss: 41.6383 - count_output_accuracy: 0.6146 - duration_output_mae: 41.6383 - val_loss: 8.8274 - val_count_output_loss: 2.7743 - val_duration_output_loss: 5.2776 - val_count_output_accuracy: 0.5387 - val_duration_output_mae: 5.2776
Epoch 2/10
286/286 [==============================] - 31s 108ms/step - loss: 25.8319 - count_output_loss: 1.5904 - duration_output_loss: 23.4110 - count_output_accuracy: 0.6772 - duration_output_mae: 23.4110 - val_loss: 2.5352 - val_count_output_loss: 0.9274 - val_duration_output_loss: 0.7795 - val_count_output_accuracy: 0.7177 - val_duration_output_mae: 0.7795
Epoch 3/10
286/286 [==============================] - 25s 88ms/step - loss: 6.4152 - count_output_loss: 1.0955 - duration_output_loss: 4.5647 - count_output_accuracy: 0.7487 - duration_output_mae: 4.5647 - val_loss: 2.0029 - val_count_output_loss: 0.7477 - val_duration_output_loss: 0

2026-05-12 23:15:04.861834: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 28s 86ms/step - loss: 49.9868 - count_output_loss: 3.4728 - duration_output_loss: 45.8376 - count_output_accuracy: 0.5841 - duration_output_mae: 45.8376 - val_loss: 7.6057 - val_count_output_loss: 3.4312 - val_duration_output_loss: 3.3410 - val_count_output_accuracy: 0.5904 - val_duration_output_mae: 3.3410
Epoch 2/10
286/286 [==============================] - 26s 89ms/step - loss: 32.2569 - count_output_loss: 1.5069 - duration_output_loss: 29.8861 - count_output_accuracy: 0.6673 - duration_output_mae: 29.8861 - val_loss: 3.2806 - val_count_output_loss: 0.9582 - val_duration_output_loss: 1.4541 - val_count_output_accuracy: 0.7571 - val_duration_output_mae: 1.4541
Epoch 3/10
286/286 [==============================] - 25s 87ms/step - loss: 8.2444 - count_output_loss: 1.2616 - duration_output_loss: 6.1791 - count_output_accuracy: 0.7081 - duration_output_mae: 6.1791 - val_loss: 2.1551 - val_count_output_loss: 0.7907 - val_duration_output_loss: 0.

...

286/286 [==============================] - 25s 80ms/step - loss: 46.8619 - count_output_loss: 3.2875 - duration_output_loss: 42.9339 - count_output_accuracy: 0.6288 - duration_output_mae: 42.9339 - val_loss: 13.3129 - val_count_output_loss: 4.8883 - val_duration_output_loss: 7.5632 - val_count_output_accuracy: 0.5676 - val_duration_output_mae: 7.5632
Epoch 2/10
286/286 [==============================] - 24s 83ms/step - loss: 34.0896 - count_output_loss: 1.7768 - duration_output_loss: 31.3081 - count_output_accuracy: 0.6815 - duration_output_mae: 31.3081 - val_loss: 4.9947 - val_count_output_loss: 1.0851 - val_duration_output_loss: 2.8911 - val_count_output_accuracy: 0.7335 - val_duration_output_mae: 2.8911
Epoch 3/10
286/286 [==============================] - 28s 97ms/step - loss: 9.3553 - count_output_loss: 1.4384 - duration_output_loss: 7.0008 - count_output_accuracy: 0.7124 - duration_output_mae: 7.0008 - val_loss: 2.4444 - val_count_output_loss: 1.0984 - val_duration_output_loss: 0

2026-05-12 23:28:57.507823: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 37s 118ms/step - loss: 45.9646 - count_output_loss: 3.0738 - duration_output_loss: 42.2243 - count_output_accuracy: 0.6568 - duration_output_mae: 42.2243 - val_loss: 20.1629 - val_count_output_loss: 14.2420 - val_duration_output_loss: 5.0749 - val_count_output_accuracy: 0.2202 - val_duration_output_mae: 5.0749
Epoch 2/10
286/286 [==============================] - 27s 95ms/step - loss: 33.3458 - count_output_loss: 1.8216 - duration_output_loss: 30.5932 - count_output_accuracy: 0.7286 - duration_output_mae: 30.5932 - val_loss: 4.1698 - val_count_output_loss: 1.4341 - val_duration_output_loss: 1.7897 - val_count_output_accuracy: 0.8319 - val_duration_output_mae: 1.7897
Epoch 3/10
286/286 [==============================] - 29s 101ms/step - loss: 9.5105 - count_output_loss: 1.2928 - duration_output_loss: 7.3307 - count_output_accuracy: 0.7542 - duration_output_mae: 7.3307 - val_loss: 2.6674 - val_count_output_loss: 1.1399 - val_duration_output_loss

2026-05-12 23:34:15.595425: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


357/357 [==============================] - 38s 101ms/step - loss: 1.7079 - count_output_loss: 1.0387 - duration_output_loss: 0.4998 - count_output_accuracy: 0.8087 - duration_output_mae: 0.4998 - val_loss: 1.4542 - val_count_output_loss: 0.8997 - val_duration_output_loss: 0.3982 - val_count_output_accuracy: 0.8179 - val_duration_output_mae: 0.3982
Epoch 2/10
357/357 [==============================] - 37s 104ms/step - loss: 1.6081 - count_output_loss: 0.9650 - duration_output_loss: 0.4954 - count_output_accuracy: 0.8100 - duration_output_mae: 0.4954 - val_loss: 1.2535 - val_count_output_loss: 0.7056 - val_duration_output_loss: 0.4172 - val_count_output_accuracy: 0.8400 - val_duration_output_mae: 0.4172
Epoch 3/10
357/357 [==============================] - 37s 104ms/step - loss: 1.6045 - count_output_loss: 0.9726 - duration_output_loss: 0.4945 - count_output_accuracy: 0.8023 - duration_output_mae: 0.4945 - val_loss: 1.6315 - val_count_output_loss: 1.0845 - val_duration_output_loss: 0.391

In [50]:
# comparison for two head model 
# cross validation results 
avgScores_dur = np.mean([x["duration_output_mae"] for x in cvScores])
stdScores_dur = np.std([x["duration_output_mae"] for x in cvScores])

avgScores_contr = np.mean([x["count_output_accuracy"] for x in cvScores])
stdScores_contr = np.std([x["count_output_accuracy"] for x in cvScores])

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model_twohead.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model_twohead.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   
 
# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full[:,0], y_pred_train_contraction)   
accuracy_test_contraction = accuracy_score(y_test[:,0], y_pred_test_contraction)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full[:,0], y_pred_train_contraction, average='weighted')  
f1_test_contraction = f1_score(y_test[:,0], y_pred_test_contraction, average='weighted')  

# MAE 
baseline = np.mean(y_train_full[:,1])
mae_baseline = np.mean(np.abs(y_train_full[:,1] - baseline))

#y_pred_train_dur = model_twohead.predict(X_train_full) 
#y_pred_test_dur = model_twohead.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full[:,1], y_pred_train_dur)   
mae_test_dur= mean_absolute_error(y_test[:,1], y_pred_test_dur)  
  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)   
print("----------------------") 
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)


Average KFold Cross Validation Score for contraction: 0.8376192808151245
Standard Deviation KFold Cross Validation Score for contractionon: 0.019719873891607526
Average KFold Cross Validation Score for duration: 0.47501028180122373
Standard Deviation KFold Cross Validation Score for duration: 0.47501028180122373
90/90 [==============================] - 2s 15ms/step
Training Accuracy : 0.8132878151260504
Test Accuracy : 0.8098739495798319
Training F1 Score : 0.7692177936924695
Test F1 Score : 0.7643092906639849
----------------------
Baseline MAE: 0.7269633995073246
Training MAE : 0.5189747332501966
Test MAE : 0.5229996758107702


## Two Channel 

Single Head Count 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]
 

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 100 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model = CNN_model_contraction_multichan(input_shape, num_classes,feature_num)
    model.compile(
    optimizer='adam',
    loss={
        'zygo_output': 'sparse_categorical_crossentropy',
        'corr_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'zygo_output': ['accuracy'],
        'corr_output': ['accuracy']
    }
)
    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]]), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
   
    scores = model.evaluate(X_val,[y_val[:, 0], y_val[:, 1]])
    for name, value in zip(model.metrics_names, scores):
        print(f"{name}: {value:.4f}")
    
    cvScores.append([scores[3]* 100,scores[4]* 100,( scores[3] + scores[4]) / 2 * 100]) 

    k += 1 
    

model_history = model.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [ ]:
# cross validation results 
cvScores_av = [fold[2] for fold in cvScores]
cvScores_zygo =  [fold[0] for fold in cvScores]
cvScores_corr = [fold[1] for fold in cvScores]

avgScores_av = np.mean(cvScores_av,axis=0)
stdScores_av = np.std(cvScores_av)

avgScores_zygo = np.mean(cvScores_zygo,axis=0)
stdScores_zygo = np.std(cvScores_zygo)

avgScores_corr = np.mean(cvScores_corr,axis=0)
stdScores_corr = np.std(cvScores_corr)

print(f"Average KFold Cross Validation Score: {avgScores_av}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores_av}")
print("\n")

print(f"Average KFold Cross Validation Score Zygo: {avgScores_zygo}")
print(f"Standard Deviation KFold Cross Validation Score Zygo: {stdScores_zygo}")
print("\n")

print(f"Average KFold Cross Validation Score Corr: {avgScores_corr}")
print(f"Standard Deviation KFold Cross Validation Score Corr: {stdScores_corr}")
print("\n")

# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model.predict(X_test)

# Convert probabilities to class labels
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)

# true labels for corr and zygo 
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]
y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# calculate accuracy for each muscle group 
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)

# calculate f1 score for each muscle group 
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')

print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n -------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)

# average score
print("\n -------- Average --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)  

Single Head Duration 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 100 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_dur = CNN_model_duration_multichan(input_shape, num_classes,feature_num)
    model_dur.compile(optimizer=Adam(learning_rate=0.0001, clipnorm=1.0),
    loss={
        'zygo_output': 'mae',
        'corr_output': 'mae'
    },
    metrics={
        'zygo_output': ['mae'],
        'corr_output': ['mae']
    }
)
    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model_dur.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]]), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_dur.evaluate(X_val,[y_val[:, 0], y_val[:, 1]])
    for name, value in zip(model_dur.metrics_names, scores):
        print(f"{name}: {value:.4f}")
    
    cvScores.append([scores[3],scores[4],( scores[3] + scores[4]) / 2 ]) 

    k += 1 
    

model_history = model_dur.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [ ]:
model_history = model_dur.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [ ]:
# cross validation results 
# cross validation results 
cvScores_av = [fold[2] for fold in cvScores]
cvScores_zygo =  [fold[0] for fold in cvScores]
cvScores_corr = [fold[1] for fold in cvScores]

avgScores_av = np.mean(cvScores_av,axis=0)
stdScores_av = np.std(cvScores_av)

avgScores_zygo = np.mean(cvScores_zygo,axis=0)
stdScores_zygo = np.std(cvScores_zygo)

avgScores_corr = np.mean(cvScores_corr,axis=0)
stdScores_corr = np.std(cvScores_corr)

print(f"Average KFold Cross Validation Score: {avgScores_av}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores_av}")
print("\n")

print(f"Average KFold Cross Validation Score Zygo: {avgScores_zygo}")
print(f"Standard Deviation KFold Cross Validation Score Zygo: {stdScores_zygo}")
print("\n")

print(f"Average KFold Cross Validation Score Corr: {avgScores_corr}")
print(f"Standard Deviation KFold Cross Validation Score Corr: {stdScores_corr}")
print("\n")

# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model_dur.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model_dur.predict(X_test)


baseline_zygo = np.mean(y_train_full[:,0])
mae_baseline_zygo = np.mean(np.abs(y_train_full[:,0] - baseline_zygo))

baseline_corr = np.mean(y_train_full[:,1])
mae_baseline_corr = np.mean(np.abs(y_train_full[:,1] - baseline_corr))
  
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,0], y_pred_train_zygo)   
mae_test_dur_zygo = mean_absolute_error(y_test[:,0], y_pred_test_zygo)  

mae_training_dur_corr = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo)   
mae_test_dur_corr= mean_absolute_error(y_test[:,1], y_pred_test_zygo)  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)
print("----------------------") 
print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)

# Four Head 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y_contr = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])


y_dur = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])


y = np.column_stack((y_contr, y_dur))
y = y[:, [0, 2, 1, 3]]

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [ ]:
kf = KFold(n_splits=4, random_state = 42, shuffle=True) # 5 folds 
cvScores_fourhead=[]
epoch_num = 10 
k = 1
from sklearn.preprocessing import StandardScaler

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_fourhead = CNN_model_fourhead_multichan(input_shape, num_classes,feature_num)
    model_fourhead.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss={
        'zygo_count_output': 'sparse_categorical_crossentropy',
        'corr_count_output': 'sparse_categorical_crossentropy',
        'zygo_duration_output': 'mae',
        'corr_duration_output': 'mae'
    },loss_weights={
            'zygo_count_output':    1.0,
            'corr_count_output':    1.0,
            'zygo_duration_output': 0.01,
            'corr_duration_output': 0.01
        },
    
    metrics={
        'zygo_count_output': ['accuracy'],
        'corr_count_output': ['accuracy'],
        'zygo_duration_output': ['mae'],
        'corr_duration_output': ['mae']
    }
)
    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model_fourhead.fit(X_train, {
        'zygo_count_output': y_train[:, 0],
        'zygo_duration_output': y_train[:, 1],
        'corr_count_output': y_train[:, 2],
        'corr_duration_output': y_train[:, 3]
    }, epochs=epoch_num, validation_data=(
    X_val,
    {
        'zygo_count_output': y_val[:, 0],
        'zygo_duration_output': y_val[:, 1],
        'corr_count_output': y_val[:, 2],
        'corr_duration_output': y_val[:, 3]
    }
), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)

    scores = model_fourhead.evaluate(X_val,[y_val[:, 0], y_val[:, 1]])
    for name, value in zip(model_fourhead.metrics_names, scores):
        print(f"{name}: {value:.4f}")
    cvScores_fourhead.append(scores)

    k += 1 
    
#model_history_fourhead = model_fourhead.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [ ]:
kf = KFold(n_splits=4, random_state = 42, shuffle=True) # 5 folds 
cvScores_fourhead=[]
epoch_num = 10 
k = 1
from sklearn.preprocessing import StandardScaler

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_fourhead = CNN_model_fourhead_multichan(input_shape, num_classes,feature_num)
    model_fourhead.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss={
        'zygo_count_output': 'sparse_categorical_crossentropy',
        'corr_count_output': 'sparse_categorical_crossentropy',
        'zygo_duration_output': 'mae',
        'corr_duration_output': 'mae'
    },loss_weights={
            'zygo_count_output':    1.0,
            'corr_count_output':    1.0,
            'zygo_duration_output': 0.01,
            'corr_duration_output': 0.01
        },
    
    metrics={
        'zygo_count_output': ['accuracy'],
        'corr_count_output': ['accuracy'],
        'zygo_duration_output': ['mae'],
        'corr_duration_output': ['mae']
    }
)
    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model_fourhead.fit(X_train, {
        'zygo_count_output': y_train[:, 0],
        'zygo_duration_output': y_train[:, 1],
        'corr_count_output': y_train[:, 2],
        'corr_duration_output': y_train[:, 3]
    }, epochs=epoch_num, validation_data=(
    X_val,
    {
        'zygo_count_output': y_val[:, 0],
        'zygo_duration_output': y_val[:, 1],
        'corr_count_output': y_val[:, 2],
        'corr_duration_output': y_val[:, 3]
    }
), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)

    scores = model_fourhead.evaluate(X_val,[y_val[:, 0], y_val[:, 1]])
    for name, value in zip(model_fourhead.metrics_names, scores):
        print(f"{name}: {value:.4f}")
    cvScores_fourhead.append(scores)

    k += 1 
    
#model_history_fourhead = model_fourhead.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [ ]:
model_fourhead = CNN_model_fourhead_multichan(input_shape, num_classes,feature_num)
epoch_num = 10 
early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
model_fourhead.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss={
        'zygo_count_output': 'sparse_categorical_crossentropy',
        'corr_count_output': 'sparse_categorical_crossentropy',
        'zygo_duration_output': 'mae',
        'corr_duration_output': 'mae'
    },loss_weights={
            'zygo_count_output':    1.0,
            'corr_count_output':    1.0,
            'zygo_duration_output': 0.01,
            'corr_duration_output': 0.01
        },
    
    metrics={
        'zygo_count_output': ['accuracy'],
        'corr_count_output': ['accuracy'],
        'zygo_duration_output': ['mae'],
        'corr_duration_output': ['mae']
    }
)
model_history_fourhead = model_fourhead.fit(X_train_full, [
        y_train_full[:, 0],
        y_train_full[:, 1],
        y_train_full[:, 2],
        y_train_full[:, 3],
    ], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [ ]:
np.shape(y_pred_test_corr_count)
np.shape(y_test_corr)

In [ ]:
'''
# Cross-validation results
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
''' 

y_pred_train = model_fourhead.predict(X_train_full)
y_pred_test = model_fourhead.predict(X_test)

y_pred_train_zygo_count, y_pred_train_zygo_dur, y_pred_train_corr_count, y_pred_train_corr_dur = y_pred_train
y_pred_test_zygo_count, y_pred_test_zygo_dur, y_pred_test_corr_count, y_pred_test_corr_dur = y_pred_test


# baselines
baseline_zygo = np.mean(y_train_full[:,1])
baseline_corr = np.mean(y_train_full[:,3])

mae_baseline_zygo = np.mean(np.abs(y_train_full[:,1] - baseline_zygo))
mae_baseline_corr = np.mean(np.abs(y_train_full[:,3] - baseline_corr))

# MAE
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo_dur)
mae_test_dur_zygo = mean_absolute_error(y_test[:,1], y_pred_test_zygo_dur)

mae_training_dur_corr = mean_absolute_error(y_train_full[:,3], y_pred_train_corr_dur)
mae_test_dur_corr = mean_absolute_error(y_test[:,3], y_pred_test_corr_dur)

print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)

print("----------------------") 

print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)



y_pred_train_zygo_count = np.argmax(y_pred_train_zygo_count, axis=1)
y_pred_train_corr_count = np.argmax(y_pred_train_corr_count, axis=1)

y_pred_test_zygo_count = np.argmax(y_pred_test_zygo_count, axis=1)
y_pred_test_corr_count = np.argmax(y_pred_test_corr_count, axis=1)

# true labels
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 2]

y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 2]

# accuracy
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo_count)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr_count)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo_count)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr_count)

# f1
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo_count, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr_count, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo_count, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr_count, average='weighted')



print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n-------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)


print("\n-------- Average (Counts) --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)

print("\n-------- Average (Duration MAE) --------")
print("Training MAE:", (mae_training_dur_zygo + mae_training_dur_corr) / 2)
print("Test MAE:", (mae_test_dur_zygo + mae_test_dur_corr) / 2)

## Segmentation

In [ ]:
def comparator(learner, instructor):
    if len(learner) != len(instructor):
        raise AssertionError("Layer count mismatch")
    for a, b in zip(learner, instructor):
        if tuple(a) != tuple(b):
            print(colored("Test failed", attrs=['bold']))
            raise AssertionError("Error in test")
    print(colored("All tests passed!", "green"))

def summary(model):
    result = []
    for layer in model.layers:
        output_shape = getattr(layer.output, 'shape', None)
        params = layer.count_params() if hasattr(layer, 'count_params') else 0
        result.append([layer.__class__.__name__, output_shape, params])
    return result
